# 🚀 StreamerCore Autonomous Cloud Streaming Engine
### Professional, 24/7 Multi-Stream SaaS Worker for StreamerCore
---
**How this works:**
1. Click the **▶ Play** button below.
2. This worker will connect to your **StreamerCore SaaS Website** (`streamercore.vercel.app`).
3. Whenever any user or customer clicks **"Go Live Now"** on your website, this cloud engine automatically downloads the video and streams it to their YouTube channel 24/7!
4. Supports multiple simultaneous live streams across multiple YouTube channels with zero lag!

In [ ]:
#@title 🌐 Start StreamerCore SaaS Worker Engine { run: "auto" }
SAAS_DOMAIN = "https://streamercore.vercel.app" #@param {type:"string"}
WORKER_SECRET = "streamercore_2026" #@param {type:"string"}

import os
import sys
import re
import time
import json
import requests
import subprocess
import threading

print("📦 Preparing Cloud Streaming Engine dependencies...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)
subprocess.run(["pip", "install", "-q", "gdown", "yt-dlp", "requests"], check=False)

print(f"\n🟢 StreamerCore Cloud Worker is INITIALIZING...")
print(f"🔗 Connected to SaaS Hub: {SAAS_DOMAIN}")

active_processes = {} # { stream_id: subprocess.Popen }

def download_video(video_url, output_path):
    if os.path.exists(output_path):
        try: os.remove(output_path)
        except: pass

    drive_match = re.search(r'/d/([a-zA-Z0-9_-]+)', video_url) or re.search(r'id=([a-zA-Z0-9_-]+)', video_url)
    if drive_match:
        file_id = drive_match.group(1)
        print(f"   ⬇️ Downloading via Google Drive ID: {file_id}")
        subprocess.run(["gdown", "--id", file_id, "-O", output_path], check=False)
    else:
        print(f"   ⬇️ Downloading via link: {video_url}")
        subprocess.run(["gdown", "--fuzzy", video_url, "-O", output_path], check=False)

    if not os.path.exists(output_path) or os.path.getsize(output_path) < 1024:
        print("   ⚠️ Retrying download with yt-dlp fallback...")
        subprocess.run(["yt-dlp", video_url, "-o", output_path], check=False)

    return os.path.exists(output_path) and os.path.getsize(output_path) >= 1024

def stream_worker_thread(stream_id, stream_title, video_url, stream_url):
    video_file = f"video_{stream_id}.mp4"
    print(f"\n▶️ [NEW STREAM STARTING] ID: {stream_id} | Title: {stream_title}")

    # 1. Download
    success = download_video(video_url, video_file)
    if not success:
        print(f"❌ Failed to download video for stream: {stream_title}")
        try:
            requests.post(f"{SAAS_DOMAIN}/api/worker", json={
                "secret": WORKER_SECRET,
                "streamId": stream_id,
                "status": "ERROR",
                "errorMsg": "Failed to download video from link"
            })
        except: pass
        return

    file_size_mb = os.path.getsize(video_file) / (1024 * 1024)
    print(f"   ✅ Download complete ({file_size_mb:.2f} MB). Launching FFmpeg to YouTube RTMP...")

    # 2. Update status to LIVE
    try:
        requests.post(f"{SAAS_DOMAIN}/api/worker", json={
            "secret": WORKER_SECRET,
            "streamId": stream_id,
            "status": "LIVE"
        })
    except Exception as e:
        print(f"   ⚠️ Status update warning: {e}")

    ffmpeg_cmd = [
        "ffmpeg",
        "-re",
        "-stream_loop", "-1",
        "-i", video_file,
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-tune", "zerolatency",
        "-b:v", "3000k",
        "-maxrate", "3000k",
        "-bufsize", "6000k",
        "-pix_fmt", "yuv420p",
        "-g", "60",
        "-c:a", "aac",
        "-b:a", "128k",
        "-ar", "44100",
        "-flvflags", "no_duration_filesize",
        "-f", "flv",
        stream_url
    ]

    # 3. Endless Loop with Auto-Restart
    while stream_id in active_processes:
        try:
            proc = subprocess.Popen(ffmpeg_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
            active_processes[stream_id] = proc
            print(f"   🔴 STREAM IS NOW LIVE ON YOUTUBE! [ID: {stream_id}]")
            proc.wait()
            if stream_id not in active_processes:
                break
            print(f"   🔄 Stream looped or re-connecting for: {stream_title}...")
            time.sleep(2)
        except Exception as e:
            print(f"   ⚠️ Stream loop error: {e}")
            time.sleep(3)

    print(f"🛑 Stream stopped for: {stream_title}")
    if os.path.exists(video_file):
        try: os.remove(video_file)
        except: pass

print("\n🚀 WORKER DAEMON RUNNING! Listening for streams from your website 24/7...")
print("💡 You can now create streams directly on https://streamercore.vercel.app!")

while True:
    try:
        res = requests.get(f"{SAAS_DOMAIN}/api/worker?secret={WORKER_SECRET}", timeout=10)
        if res.status_code == 200:
            data = res.json()
            current_streams = data.get("streams", [])
            current_ids = set()

            for s in current_streams:
                sid = str(s["_id"])
                current_ids.add(sid)
                status = s.get("status")

                # Handle new stream
                if status == "STARTING" and sid not in active_processes:
                    active_processes[sid] = True # Mark as running
                    t = threading.Thread(target=stream_worker_thread, args=(
                        sid, s.get("title", "Stream"), s.get("driveLink"), s.get("youtubeStreamKey")
                    ), daemon=True)
                    t.start()

            # Handle deleted or stopped streams
            stopped_ids = [sid for sid in active_processes if sid not in current_ids]
            for sid in stopped_ids:
                proc = active_processes.pop(sid, None)
                if proc and hasattr(proc, "kill"):
                    try: proc.kill()
                    except: pass
                print(f"🛑 Stopped stream {sid} because it was removed from website dashboard.")

        time.sleep(5)
    except KeyboardInterrupt:
        print("\n🛑 Worker stopped by user.")
        break
    except Exception as e:
        time.sleep(5)
